# STAT 764 · Meeting 16 — Bagging and random forests

**Thursday, October 29**

On Tuesday (Meeting 15) we grew single classification trees and watched them wobble: refit a
tree on a slightly different sample of patients and its first question, its leaves, and its
predictions can all change. A model that changes that much with the sample is a model with high
**variance**.

**The question for today.** If one tree is unstable, can many trees, averaged, be stable — and
better? And once a forest of 200 trees is making the predictions, what can we still say about
*why* it predicts what it does?

Same readmission problem as Meetings 9–12: will a patient be back within 30 days?

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import logit
from stat764 import diabetes

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# The readmission data from Meetings 9-12, with Meeting 9's split by PATIENT (split 764).
enc = diabetes()
enc = enc[~enc["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])]  # death, hospice
y_all = (enc["readmitted"] == "<30").astype(int)
NUM = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["age", "gender", "race", "admission_type_id", "discharge_disposition_id",
       "admission_source_id", "A1Cresult", "max_glu_serum", "insulin", "change",
       "diabetesMed", "medical_specialty"]
X_all = enc[NUM + CAT].astype({c: str for c in CAT})
train, test = next(GroupShuffleSplit(1, test_size=0.25, random_state=764)
                   .split(X_all, y_all, groups=enc["patient_nbr"]))

# Forests of 200 trees on all 74,637 training discharges take a while, so today every model is
# fitted on a random 15,000 of the training PATIENTS (about 21,000 discharges). Every cell then
# runs in seconds. The test patients are the same as in Meetings 9-12.
rng = np.random.default_rng(16)
patients = rng.choice(enc["patient_nbr"].iloc[train].unique(), 15_000, replace=False)
fit_rows = train[np.isin(enc["patient_nbr"].iloc[train], patients)]
X_fit, y_fit = X_all.iloc[fit_rows], y_all.iloc[fit_rows].to_numpy()
groups_fit = enc["patient_nbr"].iloc[fit_rows].to_numpy()   # which patient each discharge is
X_test, y_test = X_all.iloc[test], y_all.iloc[test].to_numpy()


def prep():
    """Meeting 9's preprocessing: fill and scale the counts, one-hot the categories."""
    return ColumnTransformer([
        ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT),
    ])


def make_pipe(model):
    """Preprocessing inside the pipeline, any model at the end."""
    return Pipeline([("prep", prep()), ("model", model)])


def calibration_slope(probs, outcome):
    """Meeting 11's slope: 1 = the spread is right; below 1 = too extreme; above 1 = too timid."""
    z = logit(np.clip(probs, 1e-6, 1 - 1e-6)).reshape(-1, 1)
    return LogisticRegression(C=1e6, max_iter=1000).fit(z, outcome).coef_[0, 0]


def scores(probs, outcome=y_test):
    """One row of a comparison table: the order (AUC) and the numbers (Brier, average, slope)."""
    return {"AUC": roc_auc_score(outcome, probs), "Brier": brier_score_loss(outcome, probs),
            "average said": probs.mean(), "calibration slope": calibration_slope(probs, outcome)}


print(f"fit on {len(fit_rows):,} discharges from 15,000 patients · test {len(test):,} discharges"
      f" · readmitted {y_test.mean():.1%}")

## 1. One tree, twenty answers

Draw a **bootstrap sample** of the training discharges — as many as there are, drawn *with
replacement*, so some appear twice and some not at all — fit a tree, and do it again, twenty
times. Same data, same settings; only the luck of the draw changes.

In [ ]:
# One-hot and scale once (learned on the training sample only), so twenty trees fit in seconds.
P = prep().fit(X_fit)
Z_fit, Z_test = P.transform(X_fit), P.transform(X_test)
columns = P.get_feature_names_out()

draw = np.random.default_rng(0)
tree_probs, first_questions = [], []
for b in range(20):
    rows = draw.integers(0, len(y_fit), len(y_fit))          # a bootstrap sample of the rows
    tree = DecisionTreeClassifier(min_samples_leaf=50, random_state=b).fit(Z_fit[rows], y_fit[rows])
    tree_probs.append(tree.predict_proba(Z_test)[:, 1])        # its probability for every test patient
    first_questions.append(columns[tree.tree_.feature[0]])     # the column of its first split
tree_probs = np.array(tree_probs)                              # 20 trees x 24,706 test patients

print("first question of each tree:", pd.Series(first_questions).value_counts().to_dict())
print(f"one test patient's probability, across the 20 trees: "
      f"{tree_probs[:, 0].min():.3f} to {tree_probs[:, 0].max():.3f}")
print(f"typical spread (SD) of a patient's probability across the 20 trees: "
      f"{np.median(tree_probs.std(axis=0)):.3f}")

Every tree asks the same first question — but below it they disagree, and a single patient's
probability moves around a lot from tree to tree. That spread is the variance Tuesday's trees
showed.

## 2. Bagging: average the trees

**Bagging** (*bootstrap aggregating*) is the obvious fix: fit a tree on each of many bootstrap
samples, and predict with the **average** of their probabilities. Each tree's errors are partly
luck; averaging cancels much of the luck and keeps what the trees agree on.

In [ ]:
for k in (1, 2, 5, 10, 20):
    averaged = tree_probs[:k].mean(axis=0)                     # the average of the first k trees
    print(f"  average of {k:2} trees: test AUC {roc_auc_score(y_test, averaged):.3f}")

fig, ax = plt.subplots(figsize=(7, 2.6))
for i in range(6):                                            # six test patients
    ax.scatter(tree_probs[:, i], np.full(20, i), color="#9ab", s=18)
    ax.scatter(tree_probs[:, i].mean(), i, color="k", marker="|", s=400)
ax.set(yticks=range(6), yticklabels=[f"patient {i + 1}" for i in range(6)],
       xlabel="probability of coming back (grey: each of the 20 trees · black: their average)")
plt.show()

One tree ranks the test patients poorly; the average of twenty ranks them clearly better, and the
gain is still growing at twenty. Nothing about any single tree improved. The average did.

## 3. Random forests: make the trees disagree

Averaging helps most when the trees make **different** mistakes. Bagged trees all see every
column, so they all ask the same strong first question and tend to make the same mistakes. A
**random forest** changes one thing: at every split, each tree may only choose among a random
few of the columns (`max_features="sqrt"`: about the square root of their number). The trees are
a little worse one at a time, and much less alike.

Below, both are scikit-learn's `RandomForestClassifier` with 200 trees. "Bagging" is the forest
that may look at every column (`max_features=None`).

In [ ]:
bagged = make_pipe(RandomForestClassifier(n_estimators=200, max_features=None, min_samples_leaf=5,
                                          oob_score=True, n_jobs=-1, random_state=0)).fit(X_fit, y_fit)
forest = make_pipe(RandomForestClassifier(n_estimators=200, max_features="sqrt", min_samples_leaf=5,
                                          oob_score=True, n_jobs=-1, random_state=0)).fit(X_fit, y_fit)

for name, m in (("bagging", bagged), ("random forest", forest)):
    names = m["prep"].get_feature_names_out()
    firsts = pd.Series([names[t.tree_.feature[0]] for t in m["model"].estimators_]).value_counts()
    auc = roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])
    print(f"{name:14} test AUC {auc:.3f} · first questions of its 200 trees: "
          f"{firsts.head(3).to_dict()} …")

Every bagged tree starts with the same question; the forest's trees start with many different
ones. Less alike, so the average cancels more — and the forest ranks the test patients better.

## 4. Out-of-bag: a test set that comes free

A bootstrap sample leaves out about a third of the rows — on average (1 − 1/n)ⁿ ≈ 1/e ≈ 37%.
So every training discharge was **not** seen by about a third of the trees. Predict each
discharge with only those trees and you get an honest prediction for every training discharge,
without setting any data aside: the **out-of-bag** (OOB) prediction.

In [ ]:
rows = np.random.default_rng(1).integers(0, len(y_fit), len(y_fit))
print(f"share of training discharges left out of one bootstrap sample: "
      f"{1 - len(np.unique(rows)) / len(y_fit):.3f}")

oob_probs = forest["model"].oob_decision_function_[:, 1]      # each discharge, by trees that skipped it
print(f"forest OOB AUC (training discharges, no data set aside) {roc_auc_score(y_fit, oob_probs):.3f}")
print(f"forest test AUC (the held-out patients)                 "
      f"{roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1]):.3f}")

The free estimate and the held-out one agree. One caution before trusting it everywhere: a
bootstrap sample draws **discharges**, not patients, so a patient with three discharges can be in
a tree's sample for one and out of it for another. Studio question 2 checks whether that matters
here.

## 5. Which columns does the forest use? Two kinds of importance

- **Impurity importance** (`feature_importances_`): how much each column's splits cleaned up the
  leaves, added over all the splits of all the trees — on the **training** data. Free, and biased:
  a count with hundreds of distinct values offers far more places to split than a yes/no column,
  so it wins splits by luck as well as by signal.
- **Permutation importance**: shuffle one column on **held-out** patients, so its link to the
  outcome is broken, and see how much the AUC drops. Slower, and measured where it counts.

Neither one says what would happen to a patient if you **changed** that column. Importance is
about what the *model* uses to predict — not about what *causes* a readmission.

In [ ]:
def impurity_by_column(pipe):
    """Impurity importance, with every one-hot column added back into the column it came from."""
    names = pipe["prep"].get_feature_names_out()
    raw = pd.Series(pipe["model"].feature_importances_, index=names)
    origin = [next(c for c in NUM + CAT + ["inpatient_stays_last_year"]
                   if n.split("__", 1)[1] == c or n.split("__", 1)[1].startswith(c + "_"))
              for n in names]
    return raw.groupby(origin).sum().sort_values(ascending=False)


impurity = impurity_by_column(forest)
ax = impurity.head(10)[::-1].plot.barh(figsize=(6, 3.4), color="#9ab")
ax.set(xlabel="impurity importance (share of all the trees' splitting)",
       title="The forest's impurity importance, top 10 columns")
plt.show()

## Studio

**Everything you need is in this notebook.** Sections 1–5 above explain each idea, and each question
below lists its steps. The page `forest_intro.html` in this folder walks through the same ideas with
pictures: open it in a web browser.

| | what you practice | about |
|---|---|---|
| **Question 1** | comparing five models on the order *and* the numbers | 15 minutes |
| **Question 2** | checking the free out-of-bag estimate against cross-validation | 15 minutes |
| **Question 3** | testing an AI's reading of an importance plot | 15 minutes |

**All three questions count.** If you do not finish in class, finish at home: the notebook is due 24
hours after class.

**How to work.**

1. If you opened this notebook fresh, run everything above first (**Run All**). The questions use
   `make_pipe`, `scores`, `X_fit`, `y_fit`, `X_test`, `y_test`, `groups_fit`, the fitted `bagged` and
   `forest`, `oob_probs` and `impurity_by_column`, all made in the cells above.
2. Run the **studio setup cell** below the questions. It makes the near-copy for question 3(c).
3. Before each question, **write your guess** as a comment in its cell, and keep it when the answer
   disagrees. The Compare uses it.
4. Each question ends with one sentence (or a few) to answer. Write it in a comment or a markdown cell
   under its output.
5. Save and upload to Canvas under "Studio notebook — Day 16" by **Friday, October 30, 3:40 pm**.

**Stuck?**

| you see | what to do |
|---|---|
| `NameError` for `forest`, `bagged`, `scores` or `oob_probs` | Run the notebook from the top (Run All). |
| `NameError` for `X_fit2`, `X_test2` or `make_pipe2` | Run the studio setup cell below the questions. |
| Question 1: a forest AUC far above 0.65 | You scored the training sample. Score `X_test` against `y_test`. |
| Question 2: an "out-of-bag AUC" near 0.88 | That is `oob_score_`, which is accuracy. Use the AUC of `oob_probs` against `y_fit`. |
| `ValueError` about 24,706 and 21,352 samples | Out-of-bag probabilities belong to the training stays: score them against `y_fit`, not `y_test`. |
| Question 3: a shape or index error | Permutation importance gives one value per *original* column, in the order `NUM + CAT` (`NUM2 + CAT` with the near-copy). |
| Question 3(c): the importances did not change | Fit `make_pipe2(...)` on `X_fit2`, not the old `forest`. |

### 1. Five models, one table

- **(a)** Fit one tree on the training sample:
  `make_pipe(DecisionTreeClassifier(min_samples_leaf=50, random_state=0))`. Fit Meeting 9's model
  on the same sample: `make_pipe(LogisticRegression(max_iter=2000))`. And Meeting 9's model on
  **all** the training discharges, `X_all.iloc[train]` with `y_all.iloc[train]`, as the comparison.
- **(b)** One table, one row each for the tree, `bagged`, `forest`, the logistic regression on the
  sample, and the logistic regression on all of train: `scores(model.predict_proba(X_test)[:, 1])`
  gives AUC, Brier score, average said and calibration slope. One way to build the table, once the
  three new models are fitted:

  ```python
  models = {"one tree": tree, "bagging": bagged, "random forest": forest,
            "logistic, same sample": logistic, "logistic, all of train": logistic_all}
  table = pd.DataFrame({name: scores(m.predict_proba(X_test)[:, 1]) for name, m in models.items()}).T
  print(table.round(4))
  ```

  *Reading it:* **AUC** is the order (higher ranks the test patients better). **Brier** is the
  numbers (lower is better). **Average said** should be near the 11.1% of test patients who came back.
  **Calibration slope** is 1 when the spread of the probabilities is right: below 1, too extreme;
  above 1, too timid.

**Answer in a sentence:** does the forest beat the logistic regression — on the order, and on the
numbers — and what does the last row add?

### 2. Out-of-bag against cross-validation

- **(a)** The forest's OOB AUC, from Section 4: `roc_auc_score(y_fit, oob_probs)`.
- **(b)** Five-fold cross-validation of the same forest on the training sample, **grouped by
  patient** so no patient is in two folds:
  `cross_val_score(make_pipe(RandomForestClassifier(n_estimators=200, max_features="sqrt",
  min_samples_leaf=5, n_jobs=-1, random_state=0)), X_fit, y_fit, groups=groups_fit,
  cv=GroupKFold(5), scoring="roc_auc")`. It fits five more forests, so give it a minute or two. Print
  the mean and the smallest and largest of the five.
- **(c)** Put both beside the forest's test AUC: `roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1])`.

**Answer in a sentence:** do the three agree — and which one cost five extra fits?

### 3. An AI explains the forest — check it

An assistant was shown the importance plot above and wrote:

> *"The random forest reveals the drivers of 30-day readmission. The number of lab procedures is
> the second most important factor, nearly tied with prior inpatient stays, so hospitals should
> cut unnecessary lab testing to reduce readmissions. Where the patient is discharged to ranks
> lower still, so discharge planning matters relatively little. Because the importances add up to
> one, prior inpatient stays explain about 11% of readmission risk. And when we added a second
> record of inpatient stays to the data, the two together accounted for over 20% of the model —
> confirming that prior hospital use is the dominant cause of readmission."*

- **(a)** Before computing, mark the claims you doubt. There are at least five.
- **(b)** Permutation importance on 5,000 test patients, for the original columns:
  `sample = np.random.default_rng(1).choice(len(y_test), 5000, replace=False)`, then
  `permutation_importance(forest, X_test.iloc[sample], y_test[sample], scoring="roc_auc",
  n_repeats=3, random_state=0)`; its `.importances_mean` lines up with `NUM + CAT`. Put it beside
  `impurity_by_column(forest)`, for example as two columns of one `pd.DataFrame`.
- **(c)** The "second record": add a noisy copy of `number_inpatient` to both `X_fit` and
  `X_test` (the setup cell below makes `X_fit2`, `X_test2` and a pipeline that uses it, `make_pipe2`).
  Refit a forest with the same settings on `X_fit2`, then compare both importances for the two
  columns, and the test AUC with and without the copy.

**Answer in sentences:** name three claims in the paragraph that your numbers contradict, with the
numbers.

In [ ]:
# Studio setup for question 3(c) -- run this before you start it.
noise = np.random.default_rng(3)
X_fit2, X_test2 = X_fit.copy(), X_test.copy()
X_fit2["inpatient_stays_last_year"] = X_fit2["number_inpatient"] + noise.normal(0, 0.3, len(X_fit2))
X_test2["inpatient_stays_last_year"] = X_test2["number_inpatient"] + noise.normal(0, 0.3, len(X_test2))
NUM2 = NUM + ["inpatient_stays_last_year"]                  # the counts, plus the near-copy


def make_pipe2(model):
    """make_pipe, with the near-copy of number_inpatient among the counts."""
    return Pipeline([("prep", ColumnTransformer([
        ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUM2),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT),
    ])), ("model", model)])


print(f"correlation of the copy with number_inpatient: "
      f"{np.corrcoef(X_fit2['number_inpatient'], X_fit2['inpatient_stays_last_year'])[0, 1]:.2f}")

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

## Compare — at the start of next class

We compare answers at the start of next class (Tuesday, November 3), built from the notebooks you
upload. So leave these in your notebook, in words or as printed numbers:

1. Your guesses, written before you computed, next to what you found.
2. Question 1: the table.
3. Question 3: the three claims you rejected, each with its number.

## Exit ticket

> A forest's importance plot ranks one column first. In two or three sentences: what does that
> tell you about the column — and what does it not?

**Answer in the cell below, then save this notebook and upload it to Canvas under "Studio
notebook — Day 16". Your two studio points need this answered: a blank exit ticket earns zero.
It is not graded for being right — it is how I find out what landed.**

### Your exit ticket

*(double-click to edit, and replace this line)*

## Also in this neighborhood

📗 **ISLP §8.2.1–8.2.2** — this week's reading: bagging, out-of-bag error, and random forests,
with the variable-importance plot.

📗 **ESL §15.2–15.3** — why decorrelating the trees lowers the variance of the average, and the
out-of-bag and importance details.

📗 **Strobl, Boulesteix, Zeileis & Hothorn (2007), "Bias in random forest variable importance
measures,"** *BMC Bioinformatics* 8:25 — the many-values bias in impurity importance, studied
carefully.

🎓 **Meeting 19** — interpretation tools beyond importance (partial dependence, SHAP), and their
limits.